# Features

In [5]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
import numpy as np
import matplotlib.pyplot as plt


In [1]:
"""
gc_selection_analysis.py
========================
Reproduces the SELECTION-LEVEL analysis (Chapters 1-2 / the extended chapter):
cohort + per-patient + class-conditional feature-selection patterns, and the
per-patient performance correlations.

Inputs (same folder, or edit DATA_DIR):
  LONG_SELECTED_FEATURES.csv   patient,record,k_feat,rank,feature_index,feature_name
  seizure_annotations.csv      file,seizure_id,start_sample,end_sample,...
  EVENTS_MAIN_MODEL.csv        patient,record,event_start,event_end,detected,latency_s,...
  SEGMENTS_MAIN_MODEL.csv      patient,record,...,roc_auc,seg_sensitivity,seg_specificity,fp,...

Outputs:
  fig/fig1..fig10 .png         figures
  per_patient_full.csv         per-patient table (bands, direction, network, performance)
  proc.parquet                 tidy long table with rank weights + class (reused by ictal script)

Run:  python gc_selection_analysis.py
"""
import math
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
import networkx as nx
from scipy import stats

DATA_DIR = Path(".")
OUT = Path("."); (OUT / "fig").mkdir(exist_ok=True, parents=True)
BANDS = ["delta", "theta", "alpha", "beta", "gamma"]
CH = list(range(1, 19))
BCOL = {"delta": "#3b4cc0", "theta": "#7b9ff9", "alpha": "#9aa0a6", "beta": "#f4a15c", "gamma": "#b40426"}
plt.rcParams.update({"font.size": 11, "font.family": "DejaVu Sans"})

# ----------------------------------------------------------------------------
# 1. LOAD + RANK WEIGHTING
#    feature_name = band_source_target ; 306 directed pairs x 5 bands = 1530.
#    Each record has one budget k; features ranked 1..k (rank 1 = most important).
#    Rank weight w = (k+1-rank), normalised so every record sums to 1 (equalises k).
# ----------------------------------------------------------------------------
def load():
    df = pd.read_csv(DATA_DIR / "LONG_SELECTED_FEATURES.csv")
    parts = df.feature_name.str.rsplit("_", n=2, expand=True)
    df["band"], df["src"], df["tgt"] = parts[0], parts[1].astype(int), parts[2].astype(int)
    df["w"] = df["k_feat"] + 1 - df["rank"]
    df["w"] = df["w"] / df.groupby(["patient", "record"])["w"].transform("sum")
    ann = pd.read_csv(DATA_DIR / "seizure_annotations.csv")
    df["cls"] = np.where(df.record.isin(set(ann.file.unique())), "seizure", "nonseizure")
    return df

def chan_profile(sub):
    """Return outgoing (source) and incoming (target) weight per channel."""
    o = sub.groupby("src")["w"].sum().reindex(CH, fill_value=0)
    i = sub.groupby("tgt")["w"].sum().reindex(CH, fill_value=0)
    return o, i

# ----------------------------------------------------------------------------
# 2. COHORT: FREQUENCY BAND  (Figure 1, Table 1)
# ----------------------------------------------------------------------------
def cohort_bands(df):
    uw = df["band"].value_counts().reindex(BANDS)
    prop = uw / uw.sum() * 100
    distinct = df.drop_duplicates("feature_index").groupby("band").size().reindex(BANDS)
    meanw = df.groupby("band")["w"].mean().reindex(BANDS)
    # chi-square vs uniform availability (306 features/band)
    chi2, p = stats.chisquare(uw.values, np.full(5, uw.sum() / 5))
    tbl = pd.DataFrame({"n": uw, "share_%": prop.round(2),
                        "distinct_of_306": distinct, "coverage_%": (distinct / 306 * 100).round(1),
                        "overrep": (prop / 20).round(3), "mean_rank_w": meanw.round(5)})
    print("\n[Table 1] cohort bands  chi2=%.0f p=%.1e\n" % (chi2, p), tbl)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    ax[0].bar(BANDS, prop, color=[BCOL[b] for b in BANDS], edgecolor="k", lw=.6)
    ax[0].axhline(20, ls="--", c="k"); ax[0].set_ylabel("% of selected features"); ax[0].set_title("(a) Share per band")
    ax[1].bar(BANDS, prop / 20, color=[BCOL[b] for b in BANDS], edgecolor="k", lw=.6)
    ax[1].axhline(1, ls="--", c="k"); ax[1].set_title("(b) Over-/under-representation")
    ax[2].bar(BANDS, meanw * 1000, color=[BCOL[b] for b in BANDS], edgecolor="k", lw=.6, alpha=.85)
    ax[2].set_ylabel("mean rank-weight (x1e-3)"); ax[2].set_title("(c) Importance when selected")
    ax2 = ax[2].twinx(); ax2.plot(BANDS, distinct / 306 * 100, "ko-"); ax2.set_ylabel("distinct pairs used (%)"); ax2.set_ylim(60, 105)
    plt.tight_layout(); fig.savefig(OUT / "fig/fig1_bands.png", dpi=200, bbox_inches="tight"); plt.close()
    return tbl

# ----------------------------------------------------------------------------
# 3. COHORT: DIRECTIONALITY  (Figure 2, Table 2)
# ----------------------------------------------------------------------------
def cohort_direction(df):
    o, i = chan_profile(df)
    deg = pd.DataFrame({"out": o.round(1), "in": i.round(1)})
    deg["net"] = (deg.out - deg["in"]).round(1); deg["total"] = (deg.out + deg["in"]).round(1)
    deg["asym"] = ((deg.out - deg["in"]) / (deg.out + deg["in"])).round(3)
    r = np.corrcoef(deg.out, deg["in"])[0, 1]
    print("\n[Table 2] cohort directionality  corr(out,in)=%.3f\n" % r,
          deg.sort_values("total", ascending=False))
    order = deg.sort_values("total", ascending=False).index
    y = np.arange(18)
    fig, ax = plt.subplots(1, 2, figsize=(13, 6.2))
    ax[0].barh(y, o.reindex(order), color="#c0392b", label="outgoing", edgecolor="k", lw=.4)
    ax[0].barh(y, -i.reindex(order), color="#2471a3", label="incoming", edgecolor="k", lw=.4)
    ax[0].set_yticks(y); ax[0].set_yticklabels([f"ch{c}" for c in order]); ax[0].invert_yaxis()
    ax[0].axvline(0, c="k"); ax[0].legend(); ax[0].set_title("(a) Outgoing vs incoming weight")
    asym = deg.asym.reindex(order)
    ax[1].barh(y, asym, color=["#c0392b" if v > 0 else "#2471a3" for v in asym], edgecolor="k", lw=.4)
    ax[1].set_yticks(y); ax[1].set_yticklabels([f"ch{c}" for c in order]); ax[1].invert_yaxis()
    ax[1].axvline(0, c="k"); ax[1].set_xlim(-1, 1); ax[1].set_title("(b) Directional asymmetry")
    plt.tight_layout(); fig.savefig(OUT / "fig/fig2_direction.png", dpi=200, bbox_inches="tight"); plt.close()
    return deg, r

# ----------------------------------------------------------------------------
# 4. COHORT NETWORK  (Figure 3, Table 3) + reusable drawing helper
# ----------------------------------------------------------------------------
def draw_network(ax, edges, o, i, title, top_n=40, label_fs=11):
    net = o - i
    G = nx.DiGraph(); [G.add_node(c) for c in CH]
    top = edges.sort_values("w", ascending=False).head(top_n)
    for r in top.itertuples(): G.add_edge(r.src, r.tgt)
    pos = {c: (math.cos(math.pi/2 - 2*math.pi*(c-1)/18), math.sin(math.pi/2 - 2*math.pi*(c-1)/18)) for c in CH}
    nsz = (o + i) / (o + i).max() * 2400 + 200
    norm = mcolors.TwoSlopeNorm(vmin=net.min()-1e-6, vcenter=0, vmax=net.max()+1e-6)
    nc = nx.draw_networkx_nodes(G, pos, node_size=[nsz[c] for c in CH],
                                node_color=[net[c] for c in CH], cmap="coolwarm",
                                edgecolors="k", linewidths=1.1, ax=ax); nc.set_norm(norm)
    nx.draw_networkx_labels(G, pos, {c: str(c) for c in CH}, font_size=label_fs, font_weight="bold", ax=ax)
    wmax = top.w.max()
    for r in top.itertuples():
        ax.add_patch(FancyArrowPatch(pos[r.src], pos[r.tgt], connectionstyle="arc3,rad=0.12",
                     arrowstyle="-|>", mutation_scale=11, lw=0.4 + 3.2 * r.w / wmax,
                     color=plt.cm.Greys(0.35 + 0.6 * r.w / wmax), alpha=.8, zorder=0, shrinkA=13, shrinkB=14))
    ax.set_title(title); ax.axis("off")

def cohort_network(df, o, i):
    edges = df.groupby(["src", "tgt"])["w"].sum().reset_index()
    es = edges.sort_values("w", ascending=False).reset_index(drop=True)
    cum = es.w.cumsum() / es.w.sum()
    n50, n80 = int((cum < .5).sum()) + 1, int((cum < .8).sum()) + 1
    ew = {(r.src, r.tgt): r.w for r in es.itertuples()}
    recip = sum(min(w, ew.get((b, a), 0)) for (a, b), w in ew.items()) / sum(ew.values())
    print("\n[Table 3] top edges | edges=%d  50%%=%d  80%%=%d  reciprocity=%.3f" %
          (len(edges), n50, n80, recip)); print(es.head(10).to_string(index=False))
    fig, ax = plt.subplots(figsize=(9, 9))
    draw_network(ax, edges, o, i, "Cohort directed network of selected features (top 40 edges)")
    plt.tight_layout(); fig.savefig(OUT / "fig/fig3_network.png", dpi=200, bbox_inches="tight"); plt.close()
    return edges

# ----------------------------------------------------------------------------
# 5. COHORT FREQ x DIRECTION + separation gradient  (Figure 4)
# ----------------------------------------------------------------------------
def cohort_freqdir(df):
    bsrc = df.pivot_table(index="src", columns="band", values="w", aggfunc="sum", fill_value=0).reindex(CH).reindex(columns=BANDS)
    btgt = df.pivot_table(index="tgt", columns="band", values="w", aggfunc="sum", fill_value=0).reindex(CH).reindex(columns=BANDS)
    d = df.assign(dist=(df.src - df.tgt).abs())
    spread = d.groupby("band").apply(lambda x: np.average(x.dist, weights=x.w), include_groups=False).reindex(BANDS)
    print("\n[Fig4] mean |src-tgt| separation per band\n", spread.round(2))
    fig, ax = plt.subplots(1, 3, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1, 1.05]})
    ax[0].imshow(bsrc.values, aspect="auto", cmap="Reds"); ax[0].set_title("(a) Source weight  ch x band")
    ax[1].imshow(btgt.values, aspect="auto", cmap="Blues"); ax[1].set_title("(b) Target weight  ch x band")
    for a in ax[:2]:
        a.set_xticks(range(5)); a.set_xticklabels(BANDS, rotation=30)
        a.set_yticks(range(18)); a.set_yticklabels([f"ch{c}" for c in CH])
    ax[2].plot(BANDS, spread, "o-", color="#6a0dad", lw=2, ms=8); ax[2].axhline(6, ls="--", c="gray")
    ax[2].set_title("(c) Structural separation by band"); ax[2].set_ylabel("weighted mean |src-tgt|")
    plt.tight_layout(); fig.savefig(OUT / "fig/fig4_interaction.png", dpi=200, bbox_inches="tight"); plt.close()

# ----------------------------------------------------------------------------
# 6. PER-PATIENT metrics + consistency  (Figures 5-7,9 ; Table 5)
# ----------------------------------------------------------------------------
def per_patient(df):
    pats = sorted(df.patient.unique())
    pb = df.pivot_table(index="patient", columns="band", values="w", aggfunc="sum", fill_value=0).reindex(columns=BANDS)
    pbs = pb.div(pb.sum(1), axis=0)                       # each patient's band shares
    ent = -(pbs * np.log(pbs.replace(0, np.nan))).sum(1) / np.log(5)
    M = pbs.values
    cos = [np.dot(M[i], M[j]) / (np.linalg.norm(M[i]) * np.linalg.norm(M[j])) for i, j in combinations(range(len(pats)), 2)]
    print("\n[Table 5] band consistency: gamma top in %d/24; mean band-profile cosine %.3f"
          % ((pbs.idxmax(1) == "gamma").sum(), np.mean(cos)))
    print(pd.DataFrame({"mean": pbs.mean().round(3), "sd": pbs.std().round(3),
                        "cv": (pbs.std() / pbs.mean()).round(2)}))
    # Figure 5: per-patient band heatmap + boxplots
    order = pbs.gamma.sort_values().index
    fig, ax = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={"width_ratios": [1.5, 1]})
    im = ax[0].imshow(pbs.loc[order].values, aspect="auto", cmap="magma", vmin=0, vmax=.55)
    ax[0].set_xticks(range(5)); ax[0].set_xticklabels(BANDS); ax[0].set_yticks(range(24)); ax[0].set_yticklabels(order, fontsize=9)
    ax[0].set_title("(a) Per-patient band share"); fig.colorbar(im, ax=ax[0], fraction=.046)
    bx = ax[1].boxplot([pbs[b] for b in BANDS], tick_labels=BANDS, patch_artist=True, showmeans=True)
    for patch, b in zip(bx["boxes"], BANDS): patch.set_facecolor(BCOL[b]); patch.set_alpha(.75)
    ax[1].axhline(.2, ls="--", c="k"); ax[1].set_title("(b) Between-patient variability")
    plt.tight_layout(); fig.savefig(OUT / "fig/fig5_patient_bands.png", dpi=200, bbox_inches="tight"); plt.close()

    # per-patient directionality + network rows
    rows, netmat = [], {}
    for p in pats:
        o, i = chan_profile(df[df.patient == p]); net = o - i
        asym = (o - i) / (o + i).replace(0, np.nan)
        src2 = o.sort_values(ascending=False).index[:2].tolist()
        e = df[df.patient == p].groupby(["src", "tgt"])["w"].sum().sort_values(ascending=False)
        sep = np.average((df[df.patient == p].src - df[df.patient == p].tgt).abs(), weights=df[df.patient == p].w)
        x = np.sort(e.values); n = len(x); gini = (2 * np.arange(1, n+1) - n - 1).dot(x) / (n * x.sum())
        rows.append(dict(patient=p, top_src=int(o.idxmax()), top_src_asym=round(asym.get(int(o.idxmax()), np.nan), 3),
                         top_sink=int(i.idxmax()), top2_src_share=round(o.loc[src2].sum() / o.sum(), 3),
                         ch17_18_in_top2src=int(set(src2) & {17, 18} != set()),
                         edges_used=int(e.shape[0]), top10_share=round(e.head(10).sum() / e.sum(), 3),
                         gini=round(gini, 3), mean_sep=round(sep, 2)))
        netmat[p] = net.values
    pp = pbs.round(4).reset_index().merge(pd.DataFrame(rows), on="patient")
    pp["low_share"] = pbs[["delta", "theta"]].sum(1).values.round(4); pp["entropy"] = ent.values.round(3)
    c1718 = sum(set(chan_profile(df[df.patient == p])[0].sort_values(ascending=False).index[:2]) == {17, 18} for p in pats)
    print("  {17,18} top-2 sources in %d/24 ; ch5 top sink in %d/24"
          % (c1718, sum(int(chan_profile(df[df.patient == p])[1].idxmax()) == 5 for p in pats)))

    # Figure 6: per-patient net-flow heatmap (channel x patient, normalised per patient)
    nf = np.zeros((18, len(pats)))
    for j, p in enumerate(pats):
        o, i = chan_profile(df[df.patient == p]); net = (o - i).values; nf[:, j] = net / np.abs(net).max()
    fig, ax = plt.subplots(figsize=(13, 6))
    im = ax.imshow(nf, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_yticks(range(18)); ax.set_yticklabels([f"ch{c}" for c in CH], fontsize=9)
    ax.set_xticks(range(len(pats))); ax.set_xticklabels(pats, rotation=90, fontsize=9)
    ax.set_title("Per-patient net flow by channel (red=source, blue=sink; per-patient normalised)")
    fig.colorbar(im, ax=ax, fraction=.025, pad=.01)
    plt.tight_layout(); fig.savefig(OUT / "fig/fig6_patient_netflow.png", dpi=200, bbox_inches="tight"); plt.close()

    # typical (closest to centroid) vs outlier (farthest) on band+netflow profile
    NM = np.array([netmat[p] for p in pats])
    feat = np.hstack([pbs.loc[pats].values, NM / np.linalg.norm(NM, axis=1, keepdims=True)])
    dist = np.linalg.norm(feat - feat.mean(0), axis=1)
    typical, outlier = pats[int(dist.argmin())], pats[int(dist.argmax())]
    print("  typical=%s  outlier=%s" % (typical, outlier))

    # Figure 7: per-patient network scatter
    fig, ax = plt.subplots(figsize=(9, 6.5))
    sc = ax.scatter(pp.mean_sep, pp.top10_share, s=(pp.edges_used - 40) * 3.5, c=pp.gamma,
                    cmap="viridis", edgecolors="k", lw=.6)
    for _, r in pp.iterrows(): ax.annotate(r.patient.replace("chb", ""), (r.mean_sep, r.top10_share), fontsize=7, ha="center", va="center")
    ax.set_xlabel("mean |src-tgt| separation (local <-> long-range)")
    ax.set_ylabel("top-10 edge share (distributed <-> focal)"); fig.colorbar(sc, ax=ax, label="gamma share")
    plt.tight_layout(); fig.savefig(OUT / "fig/fig7_patient_network.png", dpi=200, bbox_inches="tight"); plt.close()

    # Figure 9: example networks (typical vs outlier)
    fig, ax = plt.subplots(1, 2, figsize=(15, 7.6))
    for a, p, tag in [(ax[0], typical, "typical"), (ax[1], outlier, "outlier")]:
        sub = df[df.patient == p]; o, i = chan_profile(sub)
        draw_network(a, sub.groupby(["src", "tgt"])["w"].sum().reset_index(), o, i,
                     f'"{tag}" patient {p}', top_n=25, label_fs=9)
    plt.tight_layout(); fig.savefig(OUT / "fig/fig9_example_nets.png", dpi=200, bbox_inches="tight"); plt.close()
    return pp

# ----------------------------------------------------------------------------
# 7. CLASS-CONDITIONAL seizure vs nonseizure  (Figure 8, Table 6)
# ----------------------------------------------------------------------------
def class_conditional(df):
    bc = df.pivot_table(index="cls", columns="band", values="w", aggfunc="sum", fill_value=0).reindex(columns=BANDS)
    bcs = bc.div(bc.sum(1), axis=0)
    enr = np.log2(bcs.loc["seizure"] / bcs.loc["nonseizure"])
    sep = {c: [np.average((df[(df.cls == c) & (df.band == b)].src - df[(df.cls == c) & (df.band == b)].tgt).abs(),
               weights=df[(df.cls == c) & (df.band == b)].w) for b in BANDS] for c in ["seizure", "nonseizure"]}
    print("\n[Table 6] class-conditional band log2 enrichment (sz/ns):\n", enr.round(3))
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.8)); x = np.arange(5); wd = .38
    ax[0].bar(x - wd/2, bcs.loc["seizure"], wd, label="seizure", color="#c0392b", edgecolor="k")
    ax[0].bar(x + wd/2, bcs.loc["nonseizure"], wd, label="nonseizure", color="#2471a3", edgecolor="k")
    ax[0].set_xticks(x); ax[0].set_xticklabels(BANDS); ax[0].legend(); ax[0].set_title("(a) Band share by class")
    ax[1].plot(BANDS, sep["seizure"], "o-", color="#c0392b", label="seizure")
    ax[1].plot(BANDS, sep["nonseizure"], "s--", color="#2471a3", label="nonseizure")
    ax[1].legend(); ax[1].set_title("(b) Freq-range gradient by class")
    esz = df[df.cls == "seizure"].groupby(["src", "tgt"]).w.sum(); esz /= esz.sum()
    ens = df[df.cls == "nonseizure"].groupby(["src", "tgt"]).w.sum(); ens /= ens.sum()
    ee = pd.DataFrame({"sz": esz, "ns": ens}).fillna(0)
    ax[2].scatter(ee.ns, ee.sz, s=14, alpha=.5, color="#555")
    lim = max(ee.sz.max(), ee.ns.max()) * 1.05; ax[2].plot([0, lim], [0, lim], "k--")
    ax[2].set_xlabel("nonseizure edge share"); ax[2].set_ylabel("seizure edge share"); ax[2].set_title("(c) Per-edge share")
    plt.tight_layout(); fig.savefig(OUT / "fig/fig8_classcond.png", dpi=200, bbox_inches="tight"); plt.close()

# ----------------------------------------------------------------------------
# 8. PERFORMANCE merge + correlations  (Figure 10, Table 7)
# ----------------------------------------------------------------------------
def performance(pp):
    ev = pd.read_csv(DATA_DIR / "EVENTS_MAIN_MODEL.csv")
    seg = pd.read_csv(DATA_DIR / "SEGMENTS_MAIN_MODEL.csv")
    evp = ev.groupby("patient").agg(n_events=("detected", "size"), event_sens=("detected", "mean"),
          med_latency=("latency_s", lambda s: s[ev.loc[s.index, "detected"] == 1].median())).reset_index()
    seg["fa_per_hour"] = seg.fp / (seg.record_length / 3600.0)
    segp = seg.groupby("patient").agg(roc_auc=("roc_auc", "mean"), seg_sens=("seg_sensitivity", "mean"),
           seg_spec=("seg_specificity", "mean"), fp_total=("fp", "sum"), fa_per_hour=("fa_per_hour", "mean")).reset_index()
    M = pp.merge(evp, on="patient").merge(segp, on="patient")
    M.to_csv(OUT / "per_patient_full.csv", index=False)
    descr = ["gamma", "low_share", "top10_share", "top2_src_share", "mean_sep", "edges_used"]
    perfm = ["event_sens", "roc_auc", "seg_sens", "fa_per_hour"]
    print("\n[Table 7] Spearman(descriptor, performance) n=24")
    for d in descr:
        print(" %-15s" % d, "  ".join("%s=%+.2f%s" % (pm[:8], *(lambda r, p: (r, "*" if p < .05 else ""))(*stats.spearmanr(M[d], M[pm]))) for pm in perfm))
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
    def sc(a, xc, yc, xl, yl, t):
        a.scatter(M[xc], M[yc], s=55, c="#2E5496", edgecolors="k", lw=.5)
        for _, r in M.iterrows(): a.annotate(r.patient.replace("chb", ""), (r[xc], r[yc]), fontsize=7)
        rr, pv = stats.spearmanr(M[xc], M[yc]); b, c = np.polyfit(M[xc], M[yc], 1)
        xs = np.linspace(M[xc].min(), M[xc].max(), 10); a.plot(xs, b * xs + c, "r--")
        a.set_xlabel(xl); a.set_ylabel(yl); a.set_title(f"{t}\nSpearman r={rr:+.2f}, p={pv:.3f}")
    sc(ax[0], "top2_src_share", "seg_sens", "top-2 source concentration", "segment sensitivity", "(a) Hub conc. vs sens.")
    sc(ax[1], "gamma", "seg_sens", "gamma share", "segment sensitivity", "(b) Gamma vs sens.")
    sc(ax[2], "edges_used", "fa_per_hour", "distinct edges used", "false alarms / hour", "(c) Spread vs FA")
    plt.tight_layout(); fig.savefig(OUT / "fig/fig10_performance.png", dpi=200, bbox_inches="tight"); plt.close()
    return M

if __name__ == "__main__":
    df = load(); df.to_parquet(OUT / "proc.parquet")
    cohort_bands(df)
    deg, _ = cohort_direction(df)
    o, i = chan_profile(df)
    cohort_network(df, o, i)
    cohort_freqdir(df)
    pp = per_patient(df)
    class_conditional(df)
    performance(pp)
    print("\nDONE -> fig/fig1..fig10, per_patient_full.csv, proc.parquet")


FileNotFoundError: [Errno 2] No such file or directory: 'LONG_SELECTED_FEATURES.csv'